In [ ]:
from llama_index.core import VectorStoreIndex
from custom_llm import OurLLM
import json
from llama_index.core import ServiceContext, set_global_service_context
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

llmR = OurLLM()
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
set_global_service_context(ServiceContext.from_defaults(llm=llmR, embed_model=embed_model, chunk_size=512, context_window = 128, num_output=512))

from llama_index.core import Document
with open(r"chunks\ramayan.json", "r") as f:
    data = json.load(f)

documents = []

for i in data:       
    document = Document(
    text = i['content'],
    metadata={
        "canto_summary": i['summary'],
        "r_id": str(i['part']) + "_" + str(i['chapter']) + "_" + str(i['prompt_number'])
    },
    excluded_llm_metadata_keys=["r_id"],
    metadata_seperator="::",
    metadata_template="{value}",
    text_template="Content: {content}, Chapter context: {metadata_str}\n",
    )
    documents.append(document)

index = VectorStoreIndex.from_documents(documents)
index.storage_context.persist(persist_dir="ramayana_512_index")

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=5, verbose=True)

In [ ]:
response = query_engine.query("Who is kumbakarna and what did he do in war?")